> **Note:** This notebook follows the style of the Citadel workshop notebooks in this folder.

# 👥 HR MCP via APIM — Publish, Validate, and Use with Microsoft Agent Framework

This workshop demonstrates the HR Model Context Protocol (MCP) server through **Azure API Management (APIM) only**. It validates the Citadel-style APIM access contract, exercises MCP protocol/tool calls, demonstrates the APIM `5 tools/call per 30 seconds` policy, and shows a Microsoft Agent Framework (MAF) configuration that uses the APIM-published MCP endpoint with delegated Authorization and APIM subscription-key headers.

## Prerequisites

- Base HR MCP infrastructure has already been deployed by `workshop/mcp-hr/scripts/deploy-hr-mcp.*`.
- APIM/access-contract assets are available under `workshop/mcp-hr/infra/`.
- Publish/update APIM with `workshop/mcp-hr/scripts/publish-hr-mcp-apim.*` when needed.
- You are logged in with Azure CLI to the selected `azd` subscription/tenant.
- Your user can acquire a delegated token for `HR_MCP_SCOPE`.

No direct Container Apps endpoint is resolved or called by this notebook.


## 0️⃣ Initialize notebook variables from `azd env` and environment variables


In [ ]:

import json, os, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
from urllib.parse import quote, urlparse, urlunparse

try:
    import requests
except ImportError as exc:
    raise RuntimeError('Install workshop dependencies first: cd workshop && uv sync') from exc

WORKSHOP_DIR = Path.cwd() if Path.cwd().name == 'workshop' else Path.cwd() / 'workshop'
if not WORKSHOP_DIR.exists():
    WORKSHOP_DIR = Path.cwd()

def run(cmd: list[str], timeout: int = 60) -> subprocess.CompletedProcess:
    return subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)

def azd_get_value(name: str) -> str | None:
    try:
        r = run(['azd', 'env', 'get-value', name], timeout=20)
    except FileNotFoundError:
        return None
    return r.stdout.strip() if r.returncode == 0 and r.stdout.strip() else None

def first_value(*values: str | None) -> str | None:
    for v in values:
        if v and v.strip():
            return v.strip()
    return None

def cfg(name: str, *fallbacks: str | None) -> str | None:
    return first_value(os.getenv(name), azd_get_value(name), *fallbacks)

def require(name: str, value: str | None) -> str:
    if not value:
        raise RuntimeError(f'Missing {name}. Set it in env/azd or run the HR MCP scripts first.')
    return value

def redact(value: str | None, keep: int = 4) -> str:
    if not value:
        return '<missing>'
    return '<redacted>' if len(value) <= keep * 2 else f'{value[:keep]}…{value[-keep:]}'

def redacted_url(url: str | None) -> str:
    if not url:
        return '<missing>'
    p = urlparse(url)
    return urlunparse(p._replace(query='', fragment=''))

AZURE_SUBSCRIPTION_ID = cfg('AZURE_SUBSCRIPTION_ID')
AZURE_RESOURCE_GROUP = cfg('AZURE_RESOURCE_GROUP')
HR_MCP_TENANT_ID = cfg('HR_MCP_TENANT_ID')
HR_MCP_SCOPE = cfg('HR_MCP_SCOPE')
HR_MCP_AUDIENCE = cfg('HR_MCP_AUDIENCE')
HR_MCP_REQUIRED_SCOPE_CLAIM = cfg('HR_MCP_REQUIRED_SCOPE_CLAIM', HR_MCP_SCOPE.rsplit('/', 1)[-1] if HR_MCP_SCOPE else None)
HR_MCP_APIM_NAME = cfg('HR_MCP_APIM_NAME', cfg('APIM_NAME'), cfg('AZURE_APIM_NAME'))
HR_MCP_APIM_RESOURCE_GROUP = cfg('HR_MCP_APIM_RESOURCE_GROUP', cfg('APIM_RESOURCE_GROUP'), cfg('AZURE_APIM_RESOURCE_GROUP'), AZURE_RESOURCE_GROUP)
HR_MCP_APIM_API_NAME = cfg('HR_MCP_APIM_API_NAME', 'hr-mcp-api')
HR_MCP_APIM_PATH = cfg('HR_MCP_APIM_PATH', 'hr-mcp')
HR_MCP_APIM_PRODUCT_ID = cfg('HR_MCP_APIM_PRODUCT_ID', 'MCP-HR-Tools-DEV')
HR_MCP_APIM_SUBSCRIPTION_NAME = cfg('HR_MCP_APIM_SUBSCRIPTION_NAME', 'MCP-HR-Tools-DEV-SUB-01')
HR_MCP_APIM_MCP_URL = cfg('HR_MCP_APIM_MCP_URL')
HR_MCP_LOG_ANALYTICS_WORKSPACE_ID = cfg('HR_MCP_LOG_ANALYTICS_WORKSPACE_ID', cfg('LOG_ANALYTICS_WORKSPACE_ID'))

print('Configuration loaded (secrets redacted):')
for k, v in {
    'AZURE_SUBSCRIPTION_ID': AZURE_SUBSCRIPTION_ID,
    'AZURE_RESOURCE_GROUP': AZURE_RESOURCE_GROUP,
    'HR_MCP_TENANT_ID': HR_MCP_TENANT_ID,
    'HR_MCP_SCOPE': HR_MCP_SCOPE,
    'HR_MCP_AUDIENCE': HR_MCP_AUDIENCE,
    'HR_MCP_APIM_NAME': HR_MCP_APIM_NAME,
    'HR_MCP_APIM_RESOURCE_GROUP': HR_MCP_APIM_RESOURCE_GROUP,
    'HR_MCP_APIM_PRODUCT_ID': HR_MCP_APIM_PRODUCT_ID,
    'HR_MCP_APIM_SUBSCRIPTION_NAME': HR_MCP_APIM_SUBSCRIPTION_NAME,
    'HR_MCP_APIM_MCP_URL': redacted_url(HR_MCP_APIM_MCP_URL),
}.items():
    print(f'  {k}: {v or "<missing>"}')


## 1️⃣ Verify Azure CLI login and subscription


In [ ]:

try:
    account_result = run(['az', 'account', 'show', '-o', 'json'], timeout=30)
except FileNotFoundError as exc:
    raise RuntimeError('Azure CLI is required for this notebook.') from exc
if account_result.returncode != 0:
    raise RuntimeError('Azure CLI is not logged in. Run `az login`, then rerun this cell.')
account = json.loads(account_result.stdout)
if AZURE_SUBSCRIPTION_ID and account.get('id') != AZURE_SUBSCRIPTION_ID:
    set_result = run(['az', 'account', 'set', '--subscription', AZURE_SUBSCRIPTION_ID], timeout=30)
    if set_result.returncode != 0:
        raise RuntimeError(set_result.stderr.strip())
    account = json.loads(run(['az', 'account', 'show', '-o', 'json'], timeout=30).stdout)
print('Azure CLI login OK')
print(f"  User: {account.get('user', {}).get('name', '<unknown>')}")
print(f"  Tenant: {account.get('tenantId')}")
print(f"  Subscription: {account.get('name')} ({account.get('id')})")


## 2️⃣ Resolve APIM MCP endpoint and access-contract settings


In [ ]:

def resolve_apim_mcp_url() -> str:
    explicit = cfg('HR_MCP_APIM_MCP_URL')
    if explicit:
        p = urlparse(explicit)
        if not p.scheme or not p.netloc:
            raise RuntimeError(f'HR_MCP_APIM_MCP_URL is not absolute: {explicit!r}')
        return explicit.rstrip('/') if p.path.rstrip('/').endswith('/mcp') else explicit.rstrip('/') + '/mcp'
    apim_name = require('HR_MCP_APIM_NAME', HR_MCP_APIM_NAME)
    apim_rg = require('HR_MCP_APIM_RESOURCE_GROUP', HR_MCP_APIM_RESOURCE_GROUP)
    r = run(['az', 'apim', 'show', '--name', apim_name, '--resource-group', apim_rg, '--query', 'gatewayUrl', '-o', 'tsv'], timeout=60)
    if r.returncode != 0 or not r.stdout.strip():
        raise RuntimeError(f'Could not resolve APIM gatewayUrl: {r.stderr.strip()}')
    return f"{r.stdout.strip().rstrip('/')}/{HR_MCP_APIM_PATH or 'hr-mcp'}/mcp"

HR_MCP_APIM_MCP_URL = resolve_apim_mcp_url()
contract = {
    'apimName': HR_MCP_APIM_NAME,
    'apimResourceGroup': HR_MCP_APIM_RESOURCE_GROUP,
    'apiName': HR_MCP_APIM_API_NAME,
    'apiPath': HR_MCP_APIM_PATH,
    'productId': HR_MCP_APIM_PRODUCT_ID,
    'subscriptionName': HR_MCP_APIM_SUBSCRIPTION_NAME,
    'tenantId': HR_MCP_TENANT_ID,
    'jwtAudience': HR_MCP_AUDIENCE,
    'requiredScope': HR_MCP_REQUIRED_SCOPE_CLAIM,
    'mcpUrl': redacted_url(HR_MCP_APIM_MCP_URL),
}
print(json.dumps(contract, indent=2))


## 3️⃣ Acquire participant delegated Entra token without printing it


In [ ]:

def acquire_participant_token() -> str:
    existing = os.getenv('HR_MCP_ACCESS_TOKEN')
    if existing:
        return existing
    scope = require('HR_MCP_SCOPE', HR_MCP_SCOPE)
    cmd = ['az', 'account', 'get-access-token', '--scope', scope, '--query', 'accessToken', '-o', 'tsv']
    if HR_MCP_TENANT_ID:
        cmd[3:3] = ['--tenant', HR_MCP_TENANT_ID]
    r = run(cmd, timeout=60)
    if r.returncode != 0 or not r.stdout.strip():
        raise RuntimeError('Could not acquire delegated token. Run az login and verify HR_MCP_SCOPE/HR_MCP_TENANT_ID.')
    return r.stdout.strip()

HR_MCP_ACCESS_TOKEN = acquire_participant_token()
print(f'Delegated token acquired: {redact(HR_MCP_ACCESS_TOKEN)}')
print('Token value is intentionally not printed.')


## 4️⃣ Create the Citadel-style APIM access contract

This cell runs `publish-hr-mcp-apim` to create the APIM MCP API, backend, Citadel product, subscription, and policies. It assumes the base HR MCP infrastructure was already deployed by `deploy-hr-mcp`. Set `RUN_APIM_PUBLISH=False` only if you want to skip deployment and just validate that the infra assets exist.


In [ ]:

# Set to False to skip deployment and only validate that infra assets exist.
RUN_APIM_PUBLISH = True
publish_script = WORKSHOP_DIR / 'mcp-hr' / 'scripts' / ('publish-hr-mcp-apim.ps1' if os.name == 'nt' else 'publish-hr-mcp-apim.sh')
if not publish_script.exists():
    raise RuntimeError(f'Publish script not found: {publish_script}')
if RUN_APIM_PUBLISH:
    cmd = ['pwsh', '-File', str(publish_script)] if os.name == 'nt' else ['bash', str(publish_script)]
    r = run(cmd, timeout=900)
    if r.stdout:
        print(r.stdout)
    if r.stderr:
        print(r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f'publish-hr-mcp-apim failed (exit {r.returncode}). See output above.')
    print('APIM access contract published. In the portal verify: API "hr-mcp-api", product "MCP-HR-Tools-DEV", subscription "MCP-HR-Tools-DEV-SUB-01".')
else:
    print('Skipped (RUN_APIM_PUBLISH=False). Set RUN_APIM_PUBLISH=True to create the APIM access contract.')
for rel in ['mcp-hr/infra/main.bicep', 'mcp-hr/infra/policies/hr-mcp-api-policy.xml', 'mcp-hr/infra/policies/hr-mcp-product-policy.xml']:
    p = WORKSHOP_DIR / rel
    print(f"{'OK' if p.exists() else 'MISSING'} {rel}")


## 5️⃣ Resolve APIM subscription key without printing it


In [ ]:

def resolve_subscription_key() -> str:
    existing = os.getenv('HR_MCP_APIM_SUBSCRIPTION_KEY')
    if existing:
        return existing
    apim_name = require('HR_MCP_APIM_NAME', HR_MCP_APIM_NAME)
    apim_rg = require('HR_MCP_APIM_RESOURCE_GROUP', HR_MCP_APIM_RESOURCE_GROUP)
    sid = require('HR_MCP_APIM_SUBSCRIPTION_NAME', HR_MCP_APIM_SUBSCRIPTION_NAME)
    r = run(['az', 'apim', 'subscription', 'show', '--resource-group', apim_rg, '--service-name', apim_name, '--sid', sid, '--query', 'primaryKey', '-o', 'tsv'], timeout=60)
    if r.returncode == 0 and r.stdout.strip():
        return r.stdout.strip()
    sub_id = AZURE_SUBSCRIPTION_ID or json.loads(run(['az', 'account', 'show', '-o', 'json']).stdout)['id']
    uri = ('https://management.azure.com/subscriptions/' + quote(sub_id, safe='') + '/resourceGroups/' + quote(apim_rg, safe='') + '/providers/Microsoft.ApiManagement/service/' + quote(apim_name, safe='') + '/subscriptions/' + quote(sid, safe='') + '/listSecrets?api-version=2024-06-01-preview')
    rest = run(['az', 'rest', '--method', 'post', '--url', uri, '--query', 'primaryKey', '-o', 'tsv'], timeout=60)
    if rest.returncode == 0 and rest.stdout.strip():
        return rest.stdout.strip()
    raise RuntimeError('Could not retrieve APIM subscription key. Set HR_MCP_APIM_SUBSCRIPTION_KEY without printing it.')

HR_MCP_APIM_SUBSCRIPTION_KEY = resolve_subscription_key()
print(f'APIM subscription key resolved: {redact(HR_MCP_APIM_SUBSCRIPTION_KEY)}')


## 6️⃣ APIM-only MCP JSON-RPC helpers


In [ ]:

MCP_HEADERS = {
    'Authorization': f'Bearer {HR_MCP_ACCESS_TOKEN}',
    'Ocp-Apim-Subscription-Key': HR_MCP_APIM_SUBSCRIPTION_KEY,
    'Content-Type': 'application/json',
    'Accept': 'application/json',
}
EXPECTED_TOOLS = {'search_employees', 'get_employee_profile', 'recommend_learning_path', 'submit_pto_request', 'update_employee_skills'}

def rpc_payload(method: str, params: dict[str, Any] | None = None, request_id: str | int | None = None) -> dict[str, Any]:
    payload = {'jsonrpc': '2.0', 'method': method}
    if request_id is not None:
        payload['id'] = request_id
    if params is not None:
        payload['params'] = params
    return payload

def mcp_post(method: str, params: dict[str, Any] | None = None, request_id: str | int | None = None, timeout: int = 30) -> requests.Response:
    return requests.post(HR_MCP_APIM_MCP_URL, headers=MCP_HEADERS, json=rpc_payload(method, params, request_id), timeout=timeout)

def parse_rpc(response: requests.Response, label: str) -> dict[str, Any]:
    if response.status_code != 200:
        raise RuntimeError(f'{label} returned HTTP {response.status_code}: {response.text[:240]}')
    payload = response.json()
    if 'error' in payload:
        raise RuntimeError(f"{label} returned JSON-RPC error: {payload['error']}")
    result = payload.get('result')
    if not isinstance(result, dict):
        raise RuntimeError(f'{label} did not return a result object: {payload}')
    return result

def decode_tool_text(result: dict[str, Any]) -> dict[str, Any]:
    content = result.get('content')
    if not isinstance(content, list) or not content:
        raise RuntimeError('MCP tool result contained no content.')
    first = content[0]
    if not isinstance(first, dict) or first.get('type') != 'text':
        raise RuntimeError(f'Unexpected MCP content shape: {content}')
    return json.loads(first.get('text') or '{}')

print(f'MCP endpoint: {redacted_url(HR_MCP_APIM_MCP_URL)}')
print('Headers prepared: Authorization=<redacted>, Ocp-Apim-Subscription-Key=<redacted>')


## 7️⃣ Test initialize, tools/list, read tool, and write tool through APIM only


In [ ]:

initialize_result = parse_rpc(mcp_post('initialize', {}, 1), 'initialize')
print('initialize OK')
print(json.dumps({'protocolVersion': initialize_result.get('protocolVersion'), 'serverInfo': initialize_result.get('serverInfo')}, indent=2))

tools_result = parse_rpc(mcp_post('tools/list', {}, 2), 'tools/list')
tool_names = {t.get('name') for t in tools_result.get('tools', []) if isinstance(t, dict)}
missing, extra = EXPECTED_TOOLS - tool_names, tool_names - EXPECTED_TOOLS
if missing or extra:
    raise RuntimeError(f'Unexpected tools. Missing={sorted(missing)} Extra={sorted(extra)}')
print('tools/list OK:', sorted(tool_names))

profile = decode_tool_text(parse_rpc(mcp_post('tools/call', {'name': 'get_employee_profile', 'arguments': {'employee_id': 'E1001'}}, 'read-profile'), 'get_employee_profile'))
print('Read tool OK:', json.dumps({'employee_id': profile.get('employee_id'), 'name': profile.get('name'), 'role': profile.get('role'), 'pto': profile.get('pto')}, indent=2))

skill_update = decode_tool_text(parse_rpc(mcp_post('tools/call', {'name': 'update_employee_skills', 'arguments': {'employee_id': 'E1001', 'skills': ['apim-workshop-validation'], 'evidence_note': 'Updated through APIM-mediated MCP workshop notebook.'}}, 'write-skills'), 'update_employee_skills'))
print('Write tool OK:', json.dumps({'employee_id': skill_update.get('skill_update', {}).get('employee_id'), 'current_skills_contains_marker': 'apim-workshop-validation' in skill_update.get('current_skills', [])}, indent=2))


## 8️⃣ Demonstrate the APIM `tools/call` rate limit

The policy should allow up to 5 rapid `tools/call` requests per participant per 30 seconds and return HTTP `429` for the excess request.


In [ ]:

WAIT_FOR_FRESH_RATE_WINDOW_SECONDS = 31
print(f'Waiting {WAIT_FOR_FRESH_RATE_WINDOW_SECONDS}s for a fresh APIM rate-limit window...')
time.sleep(WAIT_FOR_FRESH_RATE_WINDOW_SECONDS)
statuses = []
for i in range(6):
    response = mcp_post('tools/call', {'name': 'get_employee_profile', 'arguments': {'employee_id': 'E1001'}}, f'rate-limit-{i}', timeout=20)
    statuses.append(response.status_code)
    if response.status_code == 429:
        print(f"Call {i + 1}: HTTP 429 as expected. Retry-After={response.headers.get('Retry-After', '<not supplied>')}")
        break
    parse_rpc(response, f'rate-limit call {i + 1}')
    print(f'Call {i + 1}: HTTP 200')
if 429 not in statuses:
    raise RuntimeError(f'Expected 429 after more than 5 rapid tools/call requests; saw {statuses}. Confirm the product policy is attached.')
print('Interpretation: APIM rejected the excess tool call before backend execution.')


## 9️⃣ Inspect APIM request/response body logs

Use the smoke-test script or KQL. Logs should include bounded request/response snippets and must not include bearer tokens or subscription keys.


In [ ]:

print('Optional smoke tests from repository root:')
print('  cd workshop && uv run python ./mcp-hr/scripts/test-hr-mcp-apim.py --skip-rate-limit')
print('  cd workshop && uv run python ./mcp-hr/scripts/test-hr-mcp-apim.py')

kql = '''
AzureDiagnostics
| where TimeGenerated > ago(1h)
| where Category has "Gateway"
| where requestUrl_s has "/hr-mcp/mcp" or Message has "hr-mcp-apim" or trace_s has "hr-mcp-apim"
| project TimeGenerated, OperationName, requestUrl_s, responseCode_d, Message, trace_s
| order by TimeGenerated desc
| take 25
'''.strip()
print('\nKQL guidance:')
print(kql)
if HR_MCP_LOG_ANALYTICS_WORKSPACE_ID:
    r = run(['az', 'monitor', 'log-analytics', 'query', '--workspace', HR_MCP_LOG_ANALYTICS_WORKSPACE_ID, '--analytics-query', kql, '-o', 'table'], timeout=90)
    print(r.stdout if r.returncode == 0 else 'Log query did not complete; verify APIM diagnostics route gateway logs/traces to this workspace.')
else:
    print('\nSet HR_MCP_LOG_ANALYTICS_WORKSPACE_ID to run this query directly.')


## 🔟 Deploy a Foundry hosted agent that uses the HR MCP via APIM

This mirrors the workshop hosted-agent (notebook #7): we build the agent image with ACR, deploy it as a
Foundry **hosted agent** in the spoke project, and invoke it. The difference is the agent's tools come from
the **HR MCP server, published through APIM** (`hr_mcp_via_apim`), instead of local Python tools.

The agent authenticates to APIM with its **managed identity**: every request carries a fresh Entra token for
the HR MCP API (application role `Mcp.Invoke`) plus the APIM subscription key. Source: `workshop/mcp-hr/hosted-agent/`.


In [ ]:

# Resolve spoke / ACR / Foundry and HR MCP values from azd.
env_subscription_id   = require('AZURE_SUBSCRIPTION_ID', cfg('AZURE_SUBSCRIPTION_ID'))
spoke_resource_group  = require('SPOKE_RESOURCE_GROUP', cfg('SPOKE_RESOURCE_GROUP'))
spoke_foundry_account = require('SPOKE_AI_FOUNDRY_ACCOUNT_NAME', cfg('SPOKE_AI_FOUNDRY_ACCOUNT_NAME'))
spoke_foundry_project = require('SPOKE_AI_FOUNDRY_PROJECT_NAME', cfg('SPOKE_AI_FOUNDRY_PROJECT_NAME'))
spoke_acr_name        = require('SPOKE_ACR_NAME', cfg('SPOKE_ACR_NAME'))
spoke_acr_login       = require('SPOKE_ACR_LOGIN_SERVER', cfg('SPOKE_ACR_LOGIN_SERVER'))

hosted_agent_name  = cfg('HR_MCP_AGENT_NAME', 'citadel-hr-mcp-agent')
hosted_agent_tag   = cfg('HR_MCP_AGENT_IMAGE_TAG', datetime.now(timezone.utc).strftime('v%Y%m%d%H%M%S'))
hosted_agent_model = cfg('HR_MCP_AGENT_MODEL', 'Hub-HR-ChatAgent-DEV-LLM/gpt-4.1')  # same model as notebook #7
hosted_agent_image = f"{spoke_acr_login}/{hosted_agent_name}:{hosted_agent_tag}"
foundry_project_endpoint = f"https://{spoke_foundry_account}.services.ai.azure.com/api/projects/{spoke_foundry_project}"

# HR MCP (APIM) connection details the agent needs.
HR_MCP_AUDIENCE = require('HR_MCP_AUDIENCE', HR_MCP_AUDIENCE)
HR_MCP_APP_ROLE_VALUE = cfg('HR_MCP_APP_ROLE_VALUE', 'Mcp.Invoke')
HR_MCP_APP_ROLE_ID = require('HR_MCP_APP_ROLE_ID', cfg('HR_MCP_APP_ROLE_ID'))
HR_MCP_API_CLIENT_ID = require('HR_MCP_API_CLIENT_ID', cfg('HR_MCP_API_CLIENT_ID'))
HR_MCP_APIM_SUBSCRIPTION_KEY = resolve_subscription_key()  # defined earlier; never printed

agent_source_dir = (WORKSHOP_DIR / 'mcp-hr' / 'hosted-agent')
if not (agent_source_dir / 'main.py').exists():
    raise RuntimeError(f'Hosted agent source not found: {agent_source_dir}')

print('Hosted agent configuration:')
print(f'  name:   {hosted_agent_name}')
print(f'  image:  {hosted_agent_image}')
print(f'  model:  {hosted_agent_model}')
print(f'  foundry endpoint: {foundry_project_endpoint}')
print(f'  MCP endpoint:     {redacted_url(HR_MCP_APIM_MCP_URL)}')
print(f'  MCP audience:     {HR_MCP_AUDIENCE}')
print(f'  app role:         {HR_MCP_APP_ROLE_VALUE}')
print(f'  subscription key: {redact(HR_MCP_APIM_SUBSCRIPTION_KEY)}')


In [ ]:

# Mirror base images into the spoke ACR so the build does not pull from Docker Hub
# every time (avoids the anonymous pull-rate limit). Falls back to public images.
base_image_ref, uv_image_ref = 'python:3.13-slim', 'ghcr.io/astral-sh/uv:latest'
if run(['az','acr','import','--name',spoke_acr_name,'--source','docker.io/library/python:3.13-slim','--image','python:3.13-slim','--force','--only-show-errors','-o','none'], timeout=300).returncode == 0:
    base_image_ref = f'{spoke_acr_login}/python:3.13-slim'
else:
    print('Note: could not import python:3.13-slim into the spoke ACR; building from Docker Hub (may hit rate limits).')
if run(['az','acr','import','--name',spoke_acr_name,'--source','ghcr.io/astral-sh/uv:latest','--image','astral-sh/uv:latest','--force','--only-show-errors','-o','none'], timeout=300).returncode == 0:
    uv_image_ref = f'{spoke_acr_login}/astral-sh/uv:latest'
else:
    print('Note: could not import the uv image into the spoke ACR; building from GHCR.')

# Build & push the agent image with ACR (no local Docker). Build context is the source folder.
acr_build = run([
    'az', 'acr', 'build',
    '--registry', spoke_acr_name,
    '--resource-group', spoke_resource_group,
    '--image', f'{hosted_agent_name}:{hosted_agent_tag}',
    '--file', str(agent_source_dir / 'Dockerfile'),
    '--build-arg', f'BASE_IMAGE={base_image_ref}',
    '--build-arg', f'UV_IMAGE={uv_image_ref}',
    '--no-logs',
    str(agent_source_dir),
], timeout=1200)
print(acr_build.stdout[-600:] if acr_build.stdout else '')
if acr_build.returncode != 0:
    raise RuntimeError(f'ACR build failed: {acr_build.stderr.strip()[-800:]}')

# Wait for the tag to appear.
for attempt in range(24):
    tags = run(['az', 'acr', 'repository', 'show-tags', '--name', spoke_acr_name, '--repository', hosted_agent_name, '-o', 'json'], timeout=60)
    if tags.returncode == 0 and hosted_agent_tag in (tags.stdout or ''):
        print(f'Image verified in ACR: {hosted_agent_image}')
        break
    print(f'  waiting for image... ({(attempt + 1) * 10}s)')
    time.sleep(10)
else:
    raise RuntimeError('Image did not appear in ACR after 4 minutes; check ACR build logs in the portal.')


In [ ]:

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import HostedAgentDefinition, ProtocolVersionRecord, AgentProtocol
from azure.identity import AzureCliCredential

project_client = AIProjectClient(endpoint=foundry_project_endpoint, credential=AzureCliCredential(process_timeout=60), allow_preview=True)

agent_version = project_client.agents.create_version(
    agent_name=hosted_agent_name,
    definition=HostedAgentDefinition(
        container_protocol_versions=[ProtocolVersionRecord(protocol=AgentProtocol.RESPONSES, version='1.0.0')],
        cpu='1',
        memory='2Gi',
        image=hosted_agent_image,
        environment_variables={
            'AZURE_AI_MODEL_DEPLOYMENT_NAME': hosted_agent_model,
            'HR_MCP_APIM_MCP_URL': HR_MCP_APIM_MCP_URL,
            'HR_MCP_AUDIENCE': HR_MCP_AUDIENCE,
            'HR_MCP_APIM_SUBSCRIPTION_KEY': HR_MCP_APIM_SUBSCRIPTION_KEY,
            'ENABLE_INSTRUMENTATION': 'true',
            'OTEL_SERVICE_NAME': hosted_agent_name,
            'OTEL_RESOURCE_ATTRIBUTES': f'service.name={hosted_agent_name},service.namespace=citadel,deployment.environment=workshop',
        },
    ),
)
print(f'Agent version created: {agent_version.name} (version {agent_version.version}), status={agent_version.status}')


In [ ]:

max_wait_seconds, poll = 600, 15
elapsed = 0
while elapsed < max_wait_seconds:
    agent_version = project_client.agents.get_version(agent_name=hosted_agent_name, agent_version=agent_version.version)
    print(f'[{elapsed}s] status: {agent_version.status}')
    if agent_version.status == 'active':
        print('Agent is active.')
        break
    if agent_version.status in ('failed', 'error'):
        raise RuntimeError(f'Agent deployment failed: {agent_version.status}')
    time.sleep(poll); elapsed += poll
else:
    raise RuntimeError(f'Timed out waiting for agent to become active after {max_wait_seconds}s.')


In [ ]:

# Grant the agent's managed identity (a) Foundry User on the spoke account and
# (b) the HR MCP API app role 'Mcp.Invoke' so it can call the MCP through APIM.
agent_identity = agent_version.instance_identity
if agent_identity is None:
    agent_identity = project_client.agents.get(agent_name=hosted_agent_name).instance_identity
if agent_identity is None:
    raise RuntimeError('Could not resolve the hosted agent identity. Ensure the agent is active.')
agent_principal_id = agent_identity.principal_id
print(f'Agent identity principal id: {agent_principal_id}')

# (a) Foundry User on the spoke Foundry account.
foundry_scope = (f"/subscriptions/{env_subscription_id}/resourceGroups/{spoke_resource_group}"
                 f"/providers/Microsoft.CognitiveServices/accounts/{spoke_foundry_account}")
existing = run(['az','role','assignment','list','--assignee-object-id',agent_principal_id,'--role','Foundry User','--scope',foundry_scope,'--query','[0].id','-o','tsv'])
if (existing.stdout or '').strip():
    print('Foundry User already assigned.')
else:
    r = run(['az','role','assignment','create','--assignee-object-id',agent_principal_id,'--assignee-principal-type','ServicePrincipal','--role','Foundry User','--scope',foundry_scope,'--only-show-errors','-o','none'])
    print('Foundry User assigned.' if r.returncode == 0 else f'Foundry User assignment failed: {r.stderr.strip()}')

# (b) HR MCP API app role (Mcp.Invoke) -> agent MI, via Microsoft Graph appRoleAssignedTo.
api_sp_id = run(['az','ad','sp','show','--id',HR_MCP_API_CLIENT_ID,'--query','id','-o','tsv']).stdout.strip()
if not api_sp_id:
    raise RuntimeError('HR MCP API service principal not found. Re-run deploy-hr-mcp.')
existing_assignments = run(['az','rest','--method','get','--url',f'https://graph.microsoft.com/v1.0/servicePrincipals/{agent_principal_id}/appRoleAssignments','-o','json'])
already = HR_MCP_APP_ROLE_ID in (existing_assignments.stdout or '')
if already:
    print('App role Mcp.Invoke already granted to the agent identity.')
else:
    body = json.dumps({'principalId': agent_principal_id, 'resourceId': api_sp_id, 'appRoleId': HR_MCP_APP_ROLE_ID})
    r = run(['az','rest','--method','post','--url',f'https://graph.microsoft.com/v1.0/servicePrincipals/{agent_principal_id}/appRoleAssignedTo','--headers','Content-Type=application/json','--body',body,'-o','none'])
    if r.returncode == 0:
        print('App role Mcp.Invoke granted. Waiting 60s for token/RBAC propagation...')
        time.sleep(60)
    else:
        print(f'App role grant failed (may already exist): {r.stderr.strip()}')


In [ ]:

import uuid

invoke_client = AIProjectClient(endpoint=foundry_project_endpoint, credential=AzureCliCredential(process_timeout=60), allow_preview=True)
openai_client = invoke_client.get_openai_client(agent_name=hosted_agent_name)

def extract_text(response) -> str:
    err = getattr(response, 'error', None)
    if err is not None:
        code_ = getattr(err, 'code', None) or (err.get('code') if isinstance(err, dict) else None)
        msg = getattr(err, 'message', None) or (err.get('message') if isinstance(err, dict) else None)
        return f'[agent error: status={getattr(response, "status", None)} code={code_} message={msg}]'
    text = getattr(response, 'output_text', None)
    if text:
        return text
    try:
        data = response.model_dump()
    except Exception:
        data = getattr(response, '__dict__', {}) or {}
    chunks = []
    def walk(node):
        if isinstance(node, dict):
            for k in ('text','output_text','value','message','delta'):
                v = node.get(k)
                if isinstance(v, str) and v.strip():
                    chunks.append(v)
            for v in node.values():
                walk(v)
        elif isinstance(node, list):
            for it in node:
                walk(it)
    walk(data)
    seen, out = set(), []
    for c in chunks:
        c = c.strip()
        if c and c not in seen:
            seen.add(c); out.append(c)
    return '\n'.join(out)

conversation_id = str(uuid.uuid4())
print(f'Conversation ID: {conversation_id}')

prompt = ('Look up employee E1001 and summarize their profile, then recommend a learning path '
          'for the target role Engineering Manager. Use the HR MCP tools.')
print(f'\nSending: {prompt}')
resp = openai_client.responses.create(input=prompt, metadata={'conversation_id': conversation_id})
print('\nAgent response:')
print(extract_text(resp))

prompt2 = ("Add the skill 'apim-hosted-agent-demo' to employee E1001 with an evidence note "
           "'Validated via the Citadel hosted agent through APIM.' Then confirm the change.")
print(f'\nSending: {prompt2}')
resp2 = openai_client.responses.create(input=prompt2, metadata={'conversation_id': conversation_id})
print('\nAgent response:')
print(extract_text(resp2))


## 1️⃣1️⃣ HR workflow through APIM

This deterministic workflow combines employee lookup, learning recommendation, and skill update through APIM. The optional MAF natural-language invocation is disabled by default to avoid consuming rate-limit budget unexpectedly.


In [ ]:

# Wait to avoid inheriting rate-limit state from the prior demonstration.
time.sleep(31)
search = decode_tool_text(parse_rpc(mcp_post('tools/call', {'name': 'search_employees', 'arguments': {'query': 'Avery', 'department': 'Engineering'}}, 'wf-search'), 'search_employees'))
print('Employee search summary:')
print(json.dumps(search, indent=2)[:1200])

employee_id = 'E1001'
learning = decode_tool_text(parse_rpc(mcp_post('tools/call', {'name': 'recommend_learning_path', 'arguments': {'employee_id': employee_id, 'target_role': 'Engineering Manager'}}, 'wf-learning'), 'recommend_learning_path'))
print('\nLearning recommendation summary:')
print(json.dumps(learning, indent=2)[:1200])

skill_marker = f"apim-maf-workshop-{datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S')}"
skill_update = decode_tool_text(parse_rpc(mcp_post('tools/call', {'name': 'update_employee_skills', 'arguments': {'employee_id': employee_id, 'skills': [skill_marker], 'evidence_note': 'APIM-only HR workflow run from the MAF workshop notebook.'}}, 'wf-skill-update'), 'update_employee_skills'))
print('\nSkill update summary:')
print(json.dumps({'employee_id': employee_id, 'added_skill': skill_marker, 'present': skill_marker in skill_update.get('current_skills', [])}, indent=2))


## ✅ Summary

You loaded configuration, verified Azure CLI context, acquired a delegated token without printing it, validated/documented APIM publication, exercised MCP protocol and HR tools through APIM only, demonstrated `429` rate limiting, inspected APIM log guidance, configured a MAF MCP tool with APIM headers, and ran an HR workflow through APIM.
